In [1]:
!pip install requests

In [1]:
import requests
import pandas as pd

In [2]:
# Fetch live NAV from mfapi.in: GET https://api.mfapi.in/mf/125497 (HDFC Top 100 Direct). 
#Parse JSON response and save as raw CSV.

url = "https://api.mfapi.in/mf/125497"
response = requests.get(url)
response.raise_for_status()
data= response.json()

In [7]:
navdf= pd.DataFrame(data["data"])

navdf["scheme_code"] = data["meta"]["scheme_code"]
navdf["scheme_name"] = data["meta"]["scheme_name"]
navdf["fund_house"] = data["meta"]["fund_house"]


In [21]:
output = r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\raw\hdfc_top_100_direct_nav.csv"

In [22]:
navdf.to_csv(output,index=False)

In [23]:
print(navdf.head())

         date        nav  scheme_code  \
0  05-06-2026  194.75470       125497   
1  04-06-2026  194.52200       125497   
2  03-06-2026  194.08100       125497   
3  02-06-2026  193.12950       125497   
4  01-06-2026  192.31950       125497   

                                 scheme_name       fund_house  
0  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
1  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
2  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
3  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  
4  SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund  


In [3]:
##Fetch NAV for 5 key schemes: SBI Bluechip (119551), ICICI Bluechip (120503), Nippon Large Cap (118632), 
#Axis Bluechip (119092), Kotak Bluechip (120841).

import requests
import pandas as pd

scheme_codes = {
    "SBI Bluechip": 119551,
    "ICICI Bluechip": 120503,
    "Nippon Large Cap": 118632,
    "Axis Bluechip": 119092,
    "Kotak Bluechip": 120841
}

NAV_data = []

for fund_name, code in scheme_codes.items():
    url = f"https://api.mfapi.in/mf/{code}"

    try:
        response = requests.get(url)
        response.raise_for_status()

        data = response.json()

        latest_nav = data["data"][0]["nav"]
        nav_date = data["data"][0]["date"]

        NAV_data.append({
            "Fund Name": fund_name,
            "Scheme Code": code,
            "NAV": latest_nav,
            "Date": nav_date
        })

    except Exception as e:
        print(f"Error fetching {fund_name}: {e}")

df_nav = pd.DataFrame(NAV_data)

print(df_nav)


          Fund Name  Scheme Code         NAV        Date
0      SBI Bluechip       119551   105.10700  05-06-2026
1    ICICI Bluechip       120503   103.81020  05-06-2026
2  Nippon Large Cap       118632    97.24560  05-06-2026
3     Axis Bluechip       119092  6168.47550  05-06-2026
4    Kotak Bluechip       120841   246.08780  05-06-2026


In [6]:
df_nav.to_csv(
    r"C:\Users\orsc\Desktop\GitDemo\My_Capstone_Project\data\raw\latest_nav_5_funds.csv",
    index=False
)

In [16]:
### Explore fund master — print unique fund houses, categories, sub-categories, risk grades. 
# Understand AMFI scheme code structure.

fund_master= pd.read_csv("01_fund_master.csv")
fundmaster =  sorted(fund_master['fund_house'].dropna().unique())
print(f"Total Fund Houses: {len(fundmaster)}")
for amc in fundmaster:
    print(amc)

Total Fund Houses: 10
Aditya Birla Sun Life MF
Axis Mutual Fund
DSP Mutual Fund
HDFC Mutual Fund
ICICI Prudential MF
Kotak Mahindra MF
Mirae Asset MF
Nippon India MF
SBI Mutual Fund
UTI Mutual Fund


In [19]:
# Print Unique Categories

categories = sorted(fund_master['category'].dropna().unique())

print(f"Total Categories: {len(categories)}"
     )
print(categories)



Total Categories: 2
['Debt', 'Equity']


In [20]:
#Print Unique Sub-Categories

subcats = sorted(fund_master['sub_category'].dropna().unique())

print(f"Total Sub Categories: {len(subcats)}")

for s in subcats:
    print(s)


Total Sub Categories: 12
ELSS
Flexi Cap
Gilt
Index
Index/ETF
Large & Mid Cap
Large Cap
Liquid
Mid Cap
Short Duration
Small Cap
Value


In [21]:
## Print Unique Risk Grades

risk_grades = sorted(fund_master['risk_category'].dropna().unique())

print(risk_grades)



['High', 'Low', 'Moderate', 'Moderately High', 'Very High']


In [23]:
### Validate AMFI Scheme Codes

import pandas as pd

fund_master = pd.read_csv("01_fund_master.csv")
nav_history = pd.read_csv("02_nav_history.csv")

fund_master["amfi_code"] = fund_master["amfi_code"].astype(str)
nav_history["amfi_code"] = nav_history["amfi_code"].astype(str)

master_codes = set(fund_master["amfi_code"].unique())
nav_codes = set(nav_history["amfi_code"].unique())

missing_in_nav = master_codes - nav_codes
extra_in_nav = nav_codes - master_codes

print(f"Fund Master Schemes : {len(master_codes)}")
print(f"NAV History Schemes : {len(nav_codes)}")
print(f"Missing in NAV      : {len(missing_in_nav)}")
print(f"Extra in NAV        : {len(extra_in_nav)}")

Fund Master Schemes : 40
NAV History Schemes : 40
Missing in NAV      : 0
Extra in NAV        : 0
